In [2]:
from urllib.robotparser import RobotFileParser

def check_robots(url):
    """Check if scraping is allowed"""
    rp = RobotFileParser()
    rp.set_url('http://mtc-m16c.sid.inpe.br/robots.txt')
    rp.read()
    return rp.can_fetch('*', url)

if check_robots('http://mtc-m16c.sid.inpe.br/rep/8JMKD3MGP8W/3HD9RLP'):
    print("✅ Scraping allowed")
else:
    print("❌ Scraping blocked by robots.txt")

✅ Scraping allowed


In [ ]:
import json
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os

# 1. Define as tuas URLs de metadados coletadas para o Evento (exemplo para o Event 07)
# Substitui esta lista com as URLs reais encontradas na página do Event 07
urls_metadados = ["http://mtc-m16c.sid.inpe.br/rep/8JMKD3MGP8W/3HD9RLP"]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
}

# Estruturas para acumular os dados mapeados
artigos_lista = []
pessoas_dict = {}  # Chave: nome_autor, Valor: {'id_pessoa': x, 'instituicao': y}
autoria_lista = []

id_artigo_count = 1
id_pessoa_count = 1
id_autoria_count = 1

def extrair_campo(soup, name):
    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) != 2:
            continue
        key = cells[0].get_text(" ", strip=True)
        if key == name:
            return cells[1].get_text(" ", strip=True)
    return None

# Ciclo principal de raspagem (Loop)
for url in urls_metadados:
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        response.encoding = "utf-8"
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Extração básica utilizando as regras do scraping_v0.ipynb
        title = extrair_campo(soup, "Title")
        abstract = extrair_campo(soup, "Abstract")
        article_type = extrair_campo(soup, "Tertiary Type")
        language = extrair_campo(soup, "Language")
        conference_name = extrair_campo(soup, "Conference Name")
        author_raw = extrair_campo(soup, "Author")
        affiliation_raw = extrair_campo(soup, "Affiliation")
        
        # Ignorar caso não consiga ler metadados essenciais
        if not title:
            continue
            
        # Determinar a edição a partir do nome da conferência
        edition = None
        if conference_name:
            m = re.search(r",\s*(\d+)\s*\(", conference_name)
            if m:
                edition = int(m.group(1))
        
        # Padronização de strings solicitada na tarefa
        if article_type == "Full paper": article_type = "full"
        elif article_type == "Short paper": article_type = "short"
        
        if language == "en": language = "english"
        elif language == "pt": language = "portuguese"
        
        # Guardar artigo
        artigos_lista.append({
            "id_artigo": id_artigo_count,
            "titulo": title,
            "abstract": abstract,
            "n_paginas": None,  # Pode ser extraído estendendo a busca por "Page Count" se houver
            "tipo_artigo": article_type,
            "lingua": language,
            "n_edicao": edition
        })
        
        # Tratamento de Autores e Afiliações usando expressões regulares do script base
        raw_authors = re.findall(r'\d+\s+(.*?)(?=\s+\d+\s+|$)', author_raw) if author_raw else []
        raw_authors = [name.strip() for name in raw_authors]
        
        formatted_authors = [
            f"{first_name.strip()} {last_name.strip()}" if ',' in aut else aut
            for aut in raw_authors
            for last_name, first_name in [aut.split(',', 1)] if ',' in aut
        ]
        
        raw_affiliations = re.findall(r'\d+\s+(.*?)(?=\s+\d+\s+|$)', affiliation_raw) if affiliation_raw else []
        formatted_affiliations = [aff.strip().replace('\ufffd', '–').replace('', '–') for aff in raw_affiliations]
        
        # Associar autores às afiliações e popular tabelas relacionais de Pessoas e Autoria
        for idx, nome_autor in enumerate(formatted_authors):
            inst = formatted_affiliations[idx] if idx < len(formatted_affiliations) else "Desconhecida"
            
            # Evitar duplicação de pessoas
            if nome_autor not in pessoas_dict:
                pessoas_dict[nome_autor] = {
                    "id_pessoa": id_pessoa_count,
                    "nome": nome_autor,
                    "instituicao": inst
                }
                id_pessoa_current = id_pessoa_count
                id_pessoa_count += 1
            else:
                id_pessoa_current = pessoas_dict[nome_autor]["id_pessoa"]
                # Atualizar instituição se a guardada anteriormente for genérica
                if pessoas_dict[nome_autor]["instituicao"] == "Desconhecida" and inst != "Desconhecida":
                    pessoas_dict[nome_autor]["instituicao"] = inst
            
            # Guardar ligação de autoria
            autoria_lista.append({
                "id_autoria": id_autoria_count,
                "id_artigo": id_artigo_count,
                "id_pessoa": id_pessoa_current,
                "ordem_autoria": idx + 1,
                "autor_correspondente": False  # Pode ser refinado se houver flag no HTML
            })
            id_autoria_count += 1
            
        id_artigo_count += 1
        print(f"✅ Processado com sucesso: {title[:40]}...")
        
    except Exception as e:
        print(f"❌ Erro ao processar a URL {url}: {e}")

# 4. Criar DataFrames finais e exportar para tabelas CSV
df_artigos = pd.DataFrame(artigos_lista)
df_pessoas = pd.DataFrame(list(pessoas_dict.values()))
df_autoria = pd.DataFrame(autoria_lista)

# Descobrir a edição dinamicamente para gerar a tabela evento.csv
edicao_detectada = df_artigos['n_edicao'].iloc[0] if not df_artigos.empty else 7 
# Altera as informações de acordo com os dados históricos do Event 07 / Event 15
df_evento = pd.DataFrame([{
    "id_evento": 1,
    "n_edicao": edicao_detectada,
    "ano": 2005, # Ajustar conforme o ano real do evento
    "local": "Local do Evento"
}])

# Criar pasta de output e salvar
output_dir = "Event_07"
os.makedirs(output_dir, exist_ok=True)

df_artigos.to_csv(f"{output_dir}/artigos.csv", index=False)
df_pessoas.to_csv(f"{output_dir}/pessoas.csv", index=False)
df_autoria.to_csv(f"{output_dir}/autoria.csv", index=False)
df_evento.to_csv(f"{output_dir}/evento.csv", index=False)

print(f"\n🎉 Concluído! Arquivos salvos na pasta '{output_dir}'.")

In [ ]:
import json
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import csv

# =====================================================================
# CONFIGURAÇÃO DE LINKS (Substitua pelos links reais que você copiou)
# =====================================================================
# Cole aqui os links de "metadata" de cada artigo do Evento 07
links_evento_07 = [
    "http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mtc-m18@80/2008/03.17.15.17.24/doc/mirrorget.cgi?metadatarepository=dpi.inpe.br/geoinfo@80/2006/07.11.13.39.30&choice=full&languagebutton=pt-BR",
    "http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mtc-m18@80/2008/03.17.15.17.24/doc/mirrorget.cgi?metadatarepository=dpi.inpe.br/geoinfo@80/2006/07.11.13.48.25&choice=full&languagebutton=pt-BR"
    # ... adicione os outros links do evento 07 aqui
]

# Cole aqui os links de "metadata" de cada artigo do Evento 15
links_evento_15 = [
    # "http://...", 
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
}

def get_field(soup, name):
    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) != 2:
            continue
        key = cells[0].get_text(" ", strip=True)
        if key == name:
            return cells[1].get_text(" ", strip=True)
    return None

def raspar_evento(lista_links, nome_pasta_output, ano_evento, n_edicao_padrao):
    print(f"\nIniciando raspagem para a pasta: {nome_pasta_output}...")
    
    artigos_lista = []
    pessoas_dict = {}
    autoria_lista = []
    
    id_artigo_count = 1
    id_pessoa_count = 1
    id_autoria_count = 1
    
    for url in lista_links:
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            response.encoding = "utf-8"
            soup = BeautifulSoup(response.text, "html.parser")
            
            title = get_field(soup, "Title")
            abstract = get_field(soup, "Abstract")
            article_type = get_field(soup, "Tertiary Type")
            language = get_field(soup, "Language")
            conference_name = get_field(soup, "Conference Name")
            author = get_field(soup, "Author")
            affiliations = get_field(soup, "Affiliation")
            
            if not title:
                continue
                
            edition = n_edicao_padrao
            if conference_name:
                m = re.search(r",\s*(\d+)\s*\(", conference_name)
                if m:
                    edition = int(m.group(1))
            
            if article_type == "Full paper": article_type = "full"
            elif article_type == "Short paper": article_type = "short"
            
            if language == "en": language = "english"
            elif language == "pt": language = "portuguese"
            
            artigos_lista.append({
                "id_artigo": id_artigo_count,
                "titulo": title,
                "abstract": abstract,
                "n_paginas": None,
                "tipo_artigo": article_type,
                "lingua": language,
                "n_edicao": edition
            })
            
            # Autores e Afiliações
            raw_authors = re.findall(r'\d+\s+(.*?)(?=\s+\d+\s+|$)', author) if author else []
            raw_authors = [name.strip() for name in raw_authors]
            formatted_authors = [
                f"{first_name.strip()} {last_name.strip()}" if ',' in aut else aut
                for aut in raw_authors
                for last_name, first_name in [aut.split(',', 1)] if ',' in aut
            ]
            
            raw_affiliations = re.findall(r'\d+\s+(.*?)(?=\s+\d+\s+|$)', affiliations) if affiliations else []
            formatted_affiliations = [aff.strip().replace('\ufffd', '–').replace('', '–') for aff in raw_affiliations]
            
            for idx, nome_autor in enumerate(formatted_authors):
                inst = formatted_affiliations[idx] if idx < len(formatted_affiliations) else "Desconhecida"
                
                if nome_autor not in p_dict := pessoas_dict:
                    pessoas_dict[nome_autor] = {
                        "id_pessoa": id_pessoa_count,
                        "nome": nome_autor,
                        "instituicao": inst
                    }
                    id_pessoa_current = id_pessoa_count
                    id_pessoa_count += 1
                else:
                    id_pessoa_current = pessoas_dict[nome_autor]["id_pessoa"]
                
                autoria_lista.append({
                    "id_autoria": id_autoria_count,
                    "id_artigo": id_artigo_count,
                    "id_pessoa": id_pessoa_current,
                    "ordem_autoria": idx + 1,
                    "autor_correspondente": "FALSE"
                })
                id_autoria_count += 1
                
            id_artigo_count += 1
            print(f"  Processed: {title[:30]}...")
        except Exception as e:
            print(f"  Error on URL {url}: {e}")
            
    # Salvar os arquivos no formato esperado pelo notebook de leitura
    if artigos_lista:
        os.makedirs(nome_pasta_output, exist_ok=True)
        
        df_artigos = pd.DataFrame(artigos_lista)
        df_pessoas = pd.DataFrame(list(pessoas_dict.values()))
        df_autoria = pd.DataFrame(autoria_lista)
        df_evento = pd.DataFrame([{"id_evento": 1, "n_edicao": n_edicao_padrao, "ano": ano_evento, "local": "Campos do Jordão"}])
        
        # Salvando com aspas globais (quoting=1) e utf-8-sig conforme read_data.ipynb
        df_artigos.to_csv(f"{nome_pasta_output}/artigos.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        df_pessoas.to_csv(f"{nome_pasta_output}/pessoas.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        df_autoria.to_csv(f"{nome_pasta_output}/autoria.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        df_evento.to_csv(f"{nome_pasta_output}/evento.csv", index=False, quoting=csv.QUOTE_ALL, encoding='utf-8-sig')
        print(f"🎉 Arquivos salvos com sucesso em '{nome_pasta_output}'!")

# Executar para os dois eventos designados no seu Issue #8
raspar_evento(links_evento_07, "2007", 2007, 7)
raspar_evento(links_evento_15, "2014", 2014, 15)